# 🧠 Teaching a Computer to Draw MRI Scans: An Intro to Diffusion Models

Welcome! In this notebook you're going to build and train a **diffusion model** — the
same basic idea behind AI image generators like DALL·E and Stable Diffusion — except
instead of generating pictures of cats or landscapes, you'll train it on real **MRI
scan images**.

### The big idea

A diffusion model learns to generate images by learning to do one very specific job:
**"given a noisy image, guess what noise was added, and remove it."**

That's it. If a model gets really good at that one job, something surprising happens:
you can start with a canvas of **pure random static** and ask the model to remove
noise from it, over and over, hundreds of times — and a brand new, realistic-looking
image slowly emerges out of the static.

We'll build this in two matching halves:

1. **The Forward Process (easy):** take a real image and gradually destroy it by
   adding a little bit of random noise, over and over, until it's pure static.
   This part has no learning in it — it's just math.
2. **The Reverse Process (hard, this is what we train):** teach a neural network to
   look at a noisy image and predict *"what noise was added here?"* If it can do
   that, we can undo the forward process, one tiny step at a time.

### Roadmap for this notebook

| Step | What we do |
|---|---|
| 1 | Install/import the tools we need |
| 2 | Download a real medical imaging dataset (MRI scans!) |
| 3 | Look at some of the data |
| 4 | Build the *forward* noise-adding process, and watch it destroy an image |
| 5 | Build a U-Net neural network — the "noise guesser" |
| 6 | Define the loss function (how the network learns) |
| 7 | Build the *reverse* denoising process |
| 8 | Train the model |
| 9 | Generate brand new, AI-imagined MRI images from pure noise! |

Let's get started.


## Step 1: Install and import our tools

We're using **PyTorch**, a popular deep learning library. The cell below installs
anything missing and imports everything we'll need throughout the notebook.

A quick note on hardware: this will run on a normal laptop CPU, but it will be slow.
If you have access to a GPU (for example on Google Colab: `Runtime > Change runtime
type > GPU`), training will be dramatically faster.


In [ ]:
# Uncomment the line below the first time you run this if a package is missing
# %pip install torch torchvision matplotlib pillow tqdm

import os
import math
import tarfile
import urllib.request

import torch
import torch.nn.functional as F
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from torchvision import transforms
from torchvision.datasets import ImageFolder

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


## Step 2: Download some real MRI data

We're going to use **MedNIST**, a free, small, education-friendly collection of
medical images released by the [MONAI](https://monai.io) project (an open-source
medical imaging AI toolkit built on PyTorch). It contains thousands of small
64×64 images pulled from several public medical imaging datasets, sorted into
categories like Chest X-Ray, Abdomen CT, Hand X-Ray, Head CT, and — the one
we want — **Breast MRI**.

We'll:
1. Download the dataset archive (~60 MB).
2. Extract it.
3. Pull out just the `BreastMRI` images and put them where our data-loading code
   expects to find them.

> This is real, de-identified medical imaging data being used purely for a
> teaching exercise. It's a nice size for a classroom: small enough to download
> and train on quickly, but real enough that students are working with
> genuine medical images rather than cartoon data.


In [ ]:
# --- Download and prepare the MedNIST "Breast MRI" images ---

MEDNIST_URL = "https://github.com/Project-MONAI/MONAI-extra-test-data/releases/download/0.8.1/MedNIST.tar.gz"
RAW_DATA_DIR = "data_raw"
ARCHIVE_PATH = os.path.join(RAW_DATA_DIR, "MedNIST.tar.gz")
EXTRACTED_DIR = os.path.join(RAW_DATA_DIR, "MedNIST")

# This is the folder structure our DataLoader (ImageFolder) expects:
# DATA_DIR / <class_name> / image1.jpeg, image2.jpeg, ...
DATA_DIR = os.path.join("data", "mri")
MRI_CLASS_DIR = os.path.join(DATA_DIR, "BreastMRI")

os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(MRI_CLASS_DIR, exist_ok=True)


def download_with_progress(url, dest_path):
    if os.path.exists(dest_path):
        print(f"Already downloaded: {dest_path}")
        return

    def report(block_num, block_size, total_size):
        downloaded = block_num * block_size
        pct = min(100, downloaded * 100 / total_size) if total_size > 0 else 0
        print(f"\rDownloading MedNIST... {pct:5.1f}%", end="")

    urllib.request.urlretrieve(url, dest_path, reporthook=report)
    print("\nDownload complete!")


# 1. Download the archive (only happens once)
download_with_progress(MEDNIST_URL, ARCHIVE_PATH)

# 2. Extract it (only happens once)
if not os.path.exists(EXTRACTED_DIR):
    print("Extracting archive...")
    with tarfile.open(ARCHIVE_PATH) as tar:
        tar.extractall(RAW_DATA_DIR)
    print("Extraction complete!")
else:
    print("Already extracted.")

# 3. Copy just the BreastMRI images into our expected folder layout
source_mri_dir = os.path.join(EXTRACTED_DIR, "BreastMRI")
if len(os.listdir(MRI_CLASS_DIR)) == 0:
    print("Copying Breast MRI images into place...")
    import shutil
    for fname in tqdm(os.listdir(source_mri_dir)):
        shutil.copy(os.path.join(source_mri_dir, fname), os.path.join(MRI_CLASS_DIR, fname))

num_images = len(os.listdir(MRI_CLASS_DIR))
print(f"\nReady! {num_images} Breast MRI images are in '{MRI_CLASS_DIR}'")


## Step 3: Load and look at the data

Neural networks don't work with images directly — they work with numbers.
`ImageFolder` below reads every image out of our `data/mri` folder and hands us
back a dataset object. The transforms do a few important jobs:

- **`RandomHorizontalFlip`**: a tiny bit of data augmentation, so the model sees
  a slightly wider variety of images (mirrored scans) during training.
- **`Resize`**: shrinks every image to a consistent 64×64 pixels. Diffusion models
  can be trained at higher resolutions, but larger images take much longer to
  train, which isn't ideal for a classroom setting.
- **`ToTensor`**: converts the image into a PyTorch tensor with pixel values
  scaled to `[0, 1]`.
- **The `Lambda`**: rescales those values to `[-1, 1]`. Diffusion models are
  typically trained on data centered around 0, which makes the math in the
  next section work out nicely.

Let's load the data and peek at a few real MRI scans.


In [ ]:
IMG_SIZE = 64
BATCH_SIZE = 64  # Smaller batch size so this comfortably trains on a laptop CPU


def load_transformed_dataset():
    data_transforms = [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),                    # Scales pixel values into [0, 1]
        transforms.Lambda(lambda t: (t * 2) - 1),  # Rescales into [-1, 1]
    ]
    data_transform = transforms.Compose(data_transforms)
    return ImageFolder(DATA_DIR, transform=data_transform)


def show_tensor_image(image):
    """Undo the transforms above so we can actually look at an image with matplotlib."""
    reverse_transforms = transforms.Compose([
        transforms.Lambda(lambda t: (t + 1) / 2),        # [-1, 1] -> [0, 1]
        transforms.Lambda(lambda t: t.permute(1, 2, 0)),  # C,H,W -> H,W,C
        transforms.Lambda(lambda t: t * 255.),
        transforms.Lambda(lambda t: t.numpy().astype(np.uint8)),
        transforms.ToPILImage(),
    ])
    if len(image.shape) == 4:  # If we got a batch, just take the first image
        image = image[0, :, :, :]
    plt.imshow(reverse_transforms(image), cmap="gray")
    plt.axis("off")


data = load_transformed_dataset()
dataloader = DataLoader(data, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f"Loaded {len(data)} MRI images.")

# Show a handful of real MRI images
plt.figure(figsize=(12, 3))
sample_batch = next(iter(dataloader))[0]
for i in range(8):
    plt.subplot(1, 8, i + 1)
    show_tensor_image(sample_batch[i])
plt.suptitle("Real Breast MRI images from our dataset")
plt.show()


## Step 4: The forward process — destroying an image with noise

Here's the core trick of diffusion models. Imagine slowly dripping a bit of random
static onto a photograph, over and over, say **300 times**. After the first drip
the photo still looks almost the same. After 150 drips it's getting hard to make
out. After all 300 drips, it's indistinguishable from pure random noise.

Mathematically, at every step `t` we mix the *original image* with some *random
noise*, controlled by a value called **beta (β)**. Beta starts small and slowly
increases — meaning we add a *little* noise early on, and *more* noise later.
This sequence of betas is called the **noise schedule**.

There's a neat mathematical shortcut here: instead of adding noise one tiny step
at a time (which would mean looping 300 times), we can jump straight from the
original image to "the image after `t` steps of noise" in a single calculation.
That's what `forward_diffusion_sample` below does — it's pure math, no neural
network involved yet.

Don't worry about memorizing the formula. The important intuition is:

> **noisy_image = (a bit of the original image) + (a bit of random noise)**

...where "a bit" is precisely controlled by the timestep `t`.


In [ ]:
def linear_beta_schedule(timesteps, start=0.0001, end=0.02):
    """The noise schedule: how much noise to add at each timestep.
    Starts small (start) and increases linearly up to (end)."""
    return torch.linspace(start, end, timesteps)


def get_index_from_list(vals, t, x_shape):
    """Helper: picks out the right schedule value for each image in a batch,
    given each image's timestep t, and reshapes it to broadcast correctly."""
    batch_size = t.shape[0]
    out = vals.gather(-1, t.cpu())
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1))).to(t.device)


def forward_diffusion_sample(x_0, t, device="cpu"):
    """Takes a clean image x_0 and a timestep t, and returns a noisy version of
    it -- as if we'd applied t steps of the forward noising process -- along
    with the exact noise that was added (we'll need that noise later to train
    the network!)."""
    noise = torch.randn_like(x_0)
    sqrt_alphas_cumprod_t = get_index_from_list(sqrt_alphas_cumprod, t, x_0.shape)
    sqrt_one_minus_alphas_cumprod_t = get_index_from_list(
        sqrt_one_minus_alphas_cumprod, t, x_0.shape
    )
    # A weighted mix of "the original image" and "pure noise"
    return (
        sqrt_alphas_cumprod_t.to(device) * x_0.to(device)
        + sqrt_one_minus_alphas_cumprod_t.to(device) * noise.to(device)
    ), noise.to(device)


# T = the total number of noising steps. 300 is a common choice: enough steps for
# smooth, gradual noising, without making training unnecessarily slow.
T = 300
betas = linear_beta_schedule(timesteps=T)

# Pre-compute a few quantities derived from the betas, so we don't recompute
# them every single time we call forward_diffusion_sample.
alphas = 1. - betas
alphas_cumprod = torch.cumprod(alphas, axis=0)
alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - alphas_cumprod)
posterior_variance = betas * (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)


### Watch the forward process in action

Let's take one real MRI image and run it through the forward process at several
different timesteps, from `t=0` (no noise) up to `t=T` (pure static). This is
exactly the target our neural network will need to learn to *reverse*.


In [ ]:
# Simulate forward diffusion on one real image
image = next(iter(dataloader))[0]

plt.figure(figsize=(15, 3))
plt.axis('off')
num_images = 10
stepsize = int(T / num_images)

for idx in range(0, T, stepsize):
    t = torch.Tensor([idx]).type(torch.int64)
    plt.subplot(1, num_images + 1, int(idx / stepsize) + 1)
    noisy_img, noise = forward_diffusion_sample(image, t)
    show_tensor_image(noisy_img)
    plt.title(f"t={idx}", fontsize=8)

plt.suptitle("Forward process: gradually destroying an MRI image with noise")
plt.show()


## Step 5: Building the "noise guesser" — a U-Net

Now for the neural network. Its job: **given a noisy image and the timestep `t`
it's currently at, predict exactly what noise was added.**

We use an architecture called a **U-Net**, named for its shape when you draw it
out — it looks like the letter U:

- The **left side (down)** repeatedly shrinks the image while extracting more and
  more abstract features (edges → shapes → textures → structures).
- The **bottom** is the most compressed, abstract representation of the image.
- The **right side (up)** mirrors the left side in reverse, growing the image back
  to full size while reconstructing detail.
- **Skip connections** copy information directly from each "down" step across to
  the matching "up" step. This helps the network keep track of fine spatial
  detail (exact edges, textures) that would otherwise get lost when the image
  is squeezed down and blown back up.

There's one more ingredient: the network also needs to know **which timestep**
it's looking at, since a slightly-noisy image (`t=10`) needs very different
treatment than an almost-pure-noise image (`t=290`). We convert the timestep
number into a vector using a `SinusoidalPositionEmbeddings` layer (the same
trick used to represent word position in Transformers), and mix that
information into every block of the U-Net.


In [ ]:
class Block(nn.Module):
    """One stage of the U-Net: two convolutions, plus a timestep embedding
    mixed in, followed by either downsampling or upsampling."""

    def __init__(self, in_ch, out_ch, time_emb_dim, up=False):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        if up:
            self.conv1 = nn.Conv2d(2 * in_ch, out_ch, 3, padding=1)
            self.transform = nn.ConvTranspose2d(out_ch, out_ch, 4, 2, 1)  # upsample
        else:
            self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
            self.transform = nn.Conv2d(out_ch, out_ch, 4, 2, 1)  # downsample
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bnorm1 = nn.BatchNorm2d(out_ch)
        self.bnorm2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU()

    def forward(self, x, t):
        h = self.bnorm1(self.relu(self.conv1(x)))       # First conv
        time_emb = self.relu(self.time_mlp(t))           # Process timestep info
        time_emb = time_emb[(..., ) + (None, ) * 2]       # Reshape to broadcast over H, W
        h = h + time_emb                                  # Mix timestep into the features
        h = self.bnorm2(self.relu(self.conv2(h)))         # Second conv
        return self.transform(h)                          # Down- or up-sample


class SinusoidalPositionEmbeddings(nn.Module):
    """Turns a timestep number (e.g. 173) into a vector the network can use,
    the same trick Transformers use to represent word position."""

    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings


class SimpleUnet(nn.Module):
    """A simplified U-Net: it takes in a noisy image + timestep, and predicts
    the noise that was added to that image."""

    def __init__(self):
        super().__init__()
        image_channels = 3
        down_channels = (64, 128, 256, 512, 1024)
        up_channels = (1024, 512, 256, 128, 64)
        out_dim = 3
        time_emb_dim = 32

        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.ReLU()
        )

        self.conv0 = nn.Conv2d(image_channels, down_channels[0], 3, padding=1)

        self.downs = nn.ModuleList([
            Block(down_channels[i], down_channels[i + 1], time_emb_dim)
            for i in range(len(down_channels) - 1)
        ])
        self.ups = nn.ModuleList([
            Block(up_channels[i], up_channels[i + 1], time_emb_dim, up=True)
            for i in range(len(up_channels) - 1)
        ])

        self.output = nn.Conv2d(up_channels[-1], out_dim, 1)

    def forward(self, x, timestep):
        t = self.time_mlp(timestep)
        x = self.conv0(x)
        residual_inputs = []
        for down in self.downs:
            x = down(x, t)
            residual_inputs.append(x)
        for up in self.ups:
            residual_x = residual_inputs.pop()
            x = torch.cat((x, residual_x), dim=1)  # Skip connection
            x = up(x, t)
        return self.output(x)


model = SimpleUnet().to(device)
print("Num params: ", sum(p.numel() for p in model.parameters()))


## Step 6: The loss function — how the network learns

Training this network is refreshingly simple once you have the pieces above:

1. Take a real image, `x_0`.
2. Pick a random timestep `t`.
3. Use the forward process to noise the image up to that timestep — this also
   gives us the *exact* noise that was added (`forward_diffusion_sample` returns
   both).
4. Ask the network to predict the noise, given the noisy image and `t`.
5. Compare the network's guess to the *actual* noise that was added, using
   **L1 loss** (basically: the average difference between the predicted and
   real noise, pixel by pixel).
6. Nudge the network's weights to make its next guess a little bit better.

Repeat that millions of times across many images and timesteps, and the network
gradually gets very good at "guess the noise" — at *every* stage of the noising
process, from barely-noisy to almost-pure-static.


In [ ]:
def get_loss(model, x_0, t):
    x_noisy, noise = forward_diffusion_sample(x_0, t, device)
    noise_pred = model(x_noisy, t)
    return F.l1_loss(noise, noise_pred)


## Step 7: The reverse process — turning noise back into an image

This is where the magic happens. Once our network is good at guessing "what noise
is in this image", we can use that ability to walk *backwards*: starting from an
image of pure random noise (`t=T`), we repeatedly ask the network "what noise do
you see?", subtract a small amount of that predicted noise, and move to the
previous timestep. Do this `T` times (from `t=299` down to `t=0`) and a coherent
image gradually emerges.

Each individual step only removes a *little* noise (plus adds a small amount of
fresh randomness, except on the very last step) — trying to jump straight from
pure noise to a final image in one shot doesn't work anywhere near as well as
many small, careful steps.


In [ ]:
@torch.no_grad()
def sample_timestep(x, t):
    """Given a noisy image x at timestep t, use the model's noise prediction
    to compute a (slightly less noisy) image at timestep t-1."""
    betas_t = get_index_from_list(betas, t, x.shape)
    sqrt_one_minus_alphas_cumprod_t = get_index_from_list(
        sqrt_one_minus_alphas_cumprod, t, x.shape
    )
    sqrt_recip_alphas_t = get_index_from_list(sqrt_recip_alphas, t, x.shape)

    # Use the model's noise prediction to estimate the mean of the denoised image
    model_mean = sqrt_recip_alphas_t * (
        x - betas_t * model(x, t) / sqrt_one_minus_alphas_cumprod_t
    )
    posterior_variance_t = get_index_from_list(posterior_variance, t, x.shape)

    if t == 0:
        # On the last step there's no more noise left to add back in
        return model_mean
    else:
        noise = torch.randn_like(x)
        return model_mean + torch.sqrt(posterior_variance_t) * noise


@torch.no_grad()
def sample_plot_image(epoch):
    """Generate one image from pure noise, showing several snapshots along
    the way, so we can watch the image 'emerge' from static."""
    img = torch.randn((1, 3, IMG_SIZE, IMG_SIZE), device=device)

    num_images = 10
    stepsize = int(T / num_images)

    fig, axes = plt.subplots(1, num_images, figsize=(15, 2))
    plot_idx = 0

    for i in range(0, T)[::-1]:
        t = torch.full((1,), i, device=device, dtype=torch.long)
        img = sample_timestep(img, t)
        img = torch.clamp(img, -1.0, 1.0)

        if i % stepsize == 0 and plot_idx < num_images:
            img_np = img.squeeze().cpu().numpy()
            img_np = img_np.transpose(1, 2, 0)
            img_np = (img_np + 1) / 2
            img_np = img_np.clip(0, 1)

            axes[plot_idx].imshow(img_np)
            axes[plot_idx].axis('off')
            plot_idx += 1

    plt.suptitle(f'Epoch {epoch}')
    plt.tight_layout()
    os.makedirs("generated", exist_ok=True)
    plt.savefig(f'generated/sample_epoch_{epoch}.png', dpi=80)
    plt.show()
    plt.close(fig)


## Step 8: Train the model

Time to put it all together. For every batch of real MRI images, we:

1. Pick a random timestep for each image in the batch.
2. Compute the loss (how good the network's noise guess was).
3. Update the network's weights to do a little better next time.

Every epoch, we'll also generate a sample image from scratch so we can *watch
the model learn* — early on the samples will look like blurry static, and they
should slowly start to resemble MRI scans as training progresses.

**Heads up:** training a diffusion model properly usually takes many hours on a
GPU. `epochs` below is set low on purpose so this notebook is practical to run
in a classroom setting — the samples won't look photorealistic, but you'll be
able to clearly see the model learning. Feel free to increase `epochs` and let
it run longer if you have the time (and ideally a GPU)!


In [ ]:
os.makedirs("generated", exist_ok=True)

optimizer = Adam(model.parameters(), lr=0.001)
epochs = 5  # Try increasing this if you have more time / a GPU!

for epoch in range(epochs):
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}")
    for step, batch in enumerate(progress_bar):
        optimizer.zero_grad()

        t = torch.randint(0, T, (batch[0].shape[0],), device=device).long()
        loss = get_loss(model, batch[0], t)
        loss.backward()
        optimizer.step()

        progress_bar.set_postfix(loss=loss.item())

    print(f"Epoch {epoch} finished | Loss: {loss.item():.4f}")
    sample_plot_image(epoch)
    torch.save(model.state_dict(), 'Model.pth')


## Step 9: Generate a brand new MRI image, from scratch

Now for the fun part! Let's generate a completely new, AI-imagined MRI image,
and save every few steps of the reverse process as frames of an animated GIF —
so you can literally watch an image emerge out of random static.

Since this model was only trained briefly (see the note in Step 8), don't expect
a perfectly realistic scan — but you should be able to see clear structure
forming, rather than pure noise. This is exactly how large-scale image
generators work under the hood; they just use far bigger networks trained for
far longer on far more data.


In [ ]:
@torch.no_grad()
def sample_to_gif(name, filename=None, fps=10):
    if filename is None:
        filename = f'generated/sample_{name}.gif'

    img = torch.randn((1, 3, IMG_SIZE, IMG_SIZE), device=device)
    num_images = 30
    stepsize = max(1, int(T / num_images))
    frames = []

    for i in range(0, T)[::-1]:
        t = torch.full((1,), i, device=device, dtype=torch.long)
        img = sample_timestep(img, t)
        img = torch.clamp(img, -1.0, 1.0)

        if i % stepsize == 0:
            img_np = img.squeeze().cpu().numpy()
            img_np = img_np.transpose(1, 2, 0)
            img_np = (img_np + 1) / 2
            img_np = (img_np.clip(0, 1) * 255).astype(np.uint8)
            frames.append(Image.fromarray(img_np))

    duration_ms = int(1000 / fps)
    frames[0].save(
        filename,
        save_all=True,
        append_images=frames[1:],
        duration=duration_ms,
        loop=0,
    )
    print(f"Saved {len(frames)}-frame GIF -> {filename}")
    return filename


gif_path = sample_to_gif(name="final")

from IPython.display import Image as IPyImage, display
display(IPyImage(filename=gif_path))


## Wrap-up & things to try

You just built and trained a real diffusion model, from the noise math all the
way to a trained U-Net generating brand-new images! Here are some ideas to
explore further:

- **Train longer.** Increase `epochs` in Step 8 — the longer the model trains,
  the sharper and more realistic its generated images should become.
- **Try a different MedNIST category.** In Step 2, instead of `"BreastMRI"`, try
  `"HeadCT"`, `"ChestCT"`, `"Hand"`, or `"AbdomenCT"` and re-run the notebook.
- **Change the noise schedule.** In Step 4, try more or fewer total timesteps
  (`T`), or a different `start`/`end` for the betas. How does that affect the
  quality of generated images?
- **Discuss:** What are some responsible-use considerations around AI-generated
  medical images? (Hint: think about how a generated "MRI" could be mistaken
  for a real one, and why that matters in a medical context.)
